# 🏥 CareFlow — Notebook 9: Resource Handover & Social Network Analysis
## Organizational Process Mining, Communication Friction & Centrality Metrics

**Author:** CareFlow Healthcare Operations Team  
**Dataset:** `careflow_analytics/output/mart_resource_handovers.parquet`

---

### Objectives:
1. Construct a clinical interaction multigraph with **NetworkX**.
2. Measure role centrality: In-Degree, Out-Degree, Betweenness, and Closeness.
3. Quantify handover communication friction (Critical $>40$m, Moderate $25-40$m, Nominal $<25$m).
4. Visualize the healthcare communication network topology.


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

handovers_df = pd.read_parquet('../careflow_analytics/output/mart_resource_handovers.parquet')
print(f"Loaded {len(handovers_df)} handover edges.")
handovers_df.head()


In [ ]:
# 1. Build Directed Network Graph
G = nx.DiGraph()

for _, row in handovers_df.iterrows():
    src = row['source_role']
    tgt = row['target_role']
    vol = row['handover_volume']
    lat = row['avg_latency_minutes']
    
    if G.has_edge(src, tgt):
        G[src][tgt]['weight'] += vol
    else:
        G.add_edge(src, tgt, weight=vol, latency=lat)

print(f"Constructed Clinical Network: {G.number_of_nodes()} staff nodes, {G.number_of_edges()} handover edges.")


In [ ]:
# 2. Centrality Analysis
in_deg = nx.in_degree_centrality(G)
out_deg = nx.out_degree_centrality(G)
betw = nx.betweenness_centrality(G, weight='weight')
pagerank = nx.pagerank(G, weight='weight')

centrality_df = pd.DataFrame({
    'In-Degree Centrality': in_deg,
    'Out-Degree Centrality': out_deg,
    'Betweenness Centrality': betw,
    'PageRank Centrality': pagerank
}).sort_values(by='Betweenness Centrality', ascending=False)

print("Staff Centrality Metrics Leaderboard:")
centrality_df


In [ ]:
# 3. Visualize Clinical Interaction Network
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42, k=1.5)

# Node sizing by PageRank
node_sizes = [pagerank[n] * 6000 for n in G.nodes()]

# Edge widths by volume
edge_weights = [G[u][v]['weight'] / 50 for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='#06b6d4', alpha=0.85)
nx.draw_networkx_edges(G, pos, width=edge_weights, edge_color='#8b5cf6', alpha=0.6, arrows=True, arrowsize=15)
nx.draw_networkx_labels(G, pos, font_size=9, font_color='#ffffff', font_weight='bold')

plt.title("Healthcare Handover Network Topology (Edge Width = Volume, Node Size = PageRank)")
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# 4. Handover Friction Breakdown
friction_counts = handovers_df['communication_friction_rating'].value_counts()
print("Friction Rating Distribution:")
print(friction_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=friction_counts.index, y=friction_counts.values, palette=['#10b981', '#f59e0b', '#f43f5e'])
plt.title("Distribution of Inter-Departmental Communication Friction")
plt.ylabel("Number of Handover Channels")
plt.tight_layout()
plt.show()
